# BTS Digital Twin - Round1 Resume From Drive

Notebook này nhận 1 link Google Drive của `gs_model/`, tự:

- tải model cũ về
- render và chấm điểm model cũ
- resume tiếp tới iteration mới
- render và chấm điểm model mới
- in chênh lệch score

Nếu muốn resume tiếp lần nữa, chỉ cần lấy `gs_model/` mới sinh ra và dùng lại notebook này.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!pip install -q gdown plyfile tqdm lpips scikit-image


## Bước 1 - Clone gaussian-splatting


In [ ]:
%cd /kaggle/working
!rm -rf gaussian-splatting
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn
import os
os.environ['GS_REPO'] = '/kaggle/working/gaussian-splatting'
print('GS_REPO =', os.environ['GS_REPO'])


## Bước 2 - Clone repo pipeline


In [ ]:
REPO_URL = 'https://github.com/ThongLuc2k3/BTS-Digital-Twin.git'
GIT_BRANCH = 'coordination/round1-status'
GITHUB_TOKEN = ''

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('Đã lấy GITHUB_TOKEN từ Kaggle Secrets')
except Exception:
    pass

clone_url = REPO_URL
if GITHUB_TOKEN and 'github.com' in REPO_URL:
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

!rm -rf /kaggle/working/project
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/project
%cd /kaggle/working/project


## Bước 3 - Tự dò dataset round1 public_set


In [ ]:
SCENE = 'hcm0031'  # hcm0031 | hcm0034 | HCM0181 | HCM0193 | HCM0204
CHECKPOINT_DRIVE_LINK = ''
TARGET_ITERATIONS = '60000'
DATASET_DRIVE_URL = 'https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link'
DATASET_ROOT_OVERRIDE = ''

assert CHECKPOINT_DRIVE_LINK, 'Chưa điền CHECKPOINT_DRIVE_LINK'

import os
from pathlib import Path
import subprocess

expected = {'hcm0031', 'hcm0034', 'HCM0181', 'HCM0193', 'HCM0204'}
candidates = []
if DATASET_ROOT_OVERRIDE:
    candidates.append(Path(DATASET_ROOT_OVERRIDE))
candidates.append(Path('/kaggle/working/project/Dataset/VAI_NVS_DATA/phase1/public_set'))

for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
    if base.exists():
        for p in base.rglob('public_set'):
            try:
                names = {x.name for x in p.iterdir() if x.is_dir()}
            except Exception:
                continue
            if expected <= names:
                candidates.append(p)

DATASET_ROOT = None
for p in candidates:
    if p.is_dir():
        names = {x.name for x in p.iterdir() if x.is_dir()}
        if expected <= names:
            DATASET_ROOT = str(p)
            break

if DATASET_ROOT is None and DATASET_DRIVE_URL:
    raw_zip = Path('/kaggle/working/dataset_round1.zip')
    raw_dir = Path('/kaggle/working/_dataset_round1_raw')
    print('Không thấy dataset mount sẵn, bắt đầu tải từ Google Drive...')
    subprocess.run(['gdown', '--fuzzy', DATASET_DRIVE_URL, '-O', str(raw_zip)], check=True)
    raw_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(raw_zip), '-d', str(raw_dir)], check=True)
    for p in raw_dir.rglob('public_set'):
        if not p.is_dir():
            continue
        try:
            names = {x.name for x in p.iterdir() if x.is_dir()}
        except Exception:
            continue
        if expected <= names:
            DATASET_ROOT = str(p)
            break

assert DATASET_ROOT, 'Không tìm thấy Dataset/VAI_NVS_DATA/phase1/public_set'
os.environ['DATASET_ROOT'] = DATASET_ROOT
print('DATASET_ROOT =', DATASET_ROOT)
print('SCENE =', SCENE)
print('TARGET_ITERATIONS =', TARGET_ITERATIONS)


## Bước 4 - Tải gs_model từ Google Drive


In [ ]:
import shutil
from pathlib import Path
import os
import re

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work' / SCENE
MODEL_DIR = WORK_DIR / 'gs_model'
RAW_DL_DIR = Path(f'/kaggle/working/_ckpt_raw/{SCENE}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
shutil.rmtree(RAW_DL_DIR, ignore_errors=True)
RAW_DL_DIR.mkdir(parents=True, exist_ok=True)
shutil.rmtree(MODEL_DIR, ignore_errors=True)

!gdown --fuzzy --folder "{CHECKPOINT_DRIVE_LINK}" -O "{RAW_DL_DIR}"
candidates = [p.parent for p in RAW_DL_DIR.rglob('cfg_args')]
assert candidates, 'Không tìm thấy cfg_args trong checkpoint tải về'
src_root = candidates[0]
shutil.copytree(src_root, MODEL_DIR)

ckpts = sorted(MODEL_DIR.glob('chkpnt*.pth'))
assert ckpts, 'Không có file chkpnt*.pth để resume'
START_CHECKPOINT = ckpts[-1]
m = re.search(r'chkpnt(\d+)\.pth$', START_CHECKPOINT.name)
assert m, f'Không parse được iteration từ {START_CHECKPOINT.name}'
BASE_ITERATION = m.group(1)
print('START_CHECKPOINT =', START_CHECKPOINT)
print('BASE_ITERATION =', BASE_ITERATION)


## Bước 5 - Chấm điểm model cũ


In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work' / SCENE
MODEL_DIR = WORK_DIR / 'gs_model'
RENDER_DIR_BEFORE = WORK_DIR / f'round1_test_renders_before_{BASE_ITERATION}'

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'render_round1_test_poses.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--model_dir', str(MODEL_DIR),
    '--iteration', BASE_ITERATION,
    '--out_dir', str(RENDER_DIR_BEFORE),
], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'eval_round1_metrics.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--renders_dir', str(RENDER_DIR_BEFORE),
    '--out_csv', str(WORK_DIR / f'eval_round1_metrics_before_{BASE_ITERATION}.csv'),
], check=True)


## Bước 6 - Resume và chấm điểm model mới


In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work' / SCENE
MODEL_DIR = WORK_DIR / 'gs_model'
RENDER_DIR_AFTER = WORK_DIR / f'round1_test_renders_after_{TARGET_ITERATIONS}'

os.environ['ITERATIONS'] = TARGET_ITERATIONS
os.environ['ANTIALIASING'] = '1'
os.environ['EXPOSURE_COMP'] = '1'
os.environ['SAVE_FINAL_CHECKPOINT'] = '1'
os.environ['START_CHECKPOINT'] = str(START_CHECKPOINT)

subprocess.run(['bash', str(PROJECT / 'pipeline' / 'scripts' / '03_train_3dgs.sh'), SCENE], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'render_round1_test_poses.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--model_dir', str(MODEL_DIR),
    '--iteration', TARGET_ITERATIONS,
    '--out_dir', str(RENDER_DIR_AFTER),
], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'eval_round1_metrics.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--renders_dir', str(RENDER_DIR_AFTER),
    '--out_csv', str(WORK_DIR / f'eval_round1_metrics_after_{TARGET_ITERATIONS}.csv'),
], check=True)


## Bước 7 - So sánh score cũ và mới


In [ ]:
import csv
from pathlib import Path

WORK_DIR = Path('/kaggle/working/project/pipeline/work') / SCENE
csv_before = WORK_DIR / f'eval_round1_metrics_before_{BASE_ITERATION}.csv'
csv_after = WORK_DIR / f'eval_round1_metrics_after_{TARGET_ITERATIONS}.csv'

def mean_score(path):
    rows = list(csv.DictReader(open(path)))
    return sum(float(r['score']) for r in rows) / len(rows)

score_before = mean_score(csv_before)
score_after = mean_score(csv_after)
print(f'Score old model ({BASE_ITERATION}): {score_before:.6f}')
print(f'Score new model ({TARGET_ITERATIONS}): {score_after:.6f}')
print(f'Chênh lệch: {score_after - score_before:+.6f}')
if score_after > score_before:
    print('Kết luận: model mới tốt hơn model cũ')
else:
    print('Kết luận: model mới chưa tốt hơn model cũ')


## Bước 8 - gs_model mới để resume tiếp

Nếu muốn resume tiếp, chỉ cần tải thư mục `gs_model/` mới này về rồi dùng lại notebook này với link Drive mới.


In [ ]:
print(f'/kaggle/working/project/pipeline/work/{SCENE}/gs_model')
